# Thermodynamic Analysis of Unidentified Photosynthetic Modes

This notebook investigates flux solutions classified as "Unidentified" (neither C3, C4, nor intermediate types) to determine their thermodynamic feasibility. 

**Workflow:**
1.  **Data Loading & Preprocessing:** Load the metabolic model and flux scan results.
2.  **Flux Normalization:** Normalize fluxes based on anatomical constraints (volume ratios).
3.  **Classification:** Filter for flux solutions that do not fit standard photosynthetic definitions.
4.  **Reaction Identification:** Identify reactions producing CO2 in these unidentified solutions.
5.  **Thermodynamic Analysis:** Use `equilibrator_api` to calculate $\Delta G'$ for candidate reactions.
6.  **Monte Carlo Sampling:** Assess thermodynamic feasibility under physiological concentration ranges.

## 1. Imports and Setup

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from cobra.io import read_sbml_model
import pandas as pd
from Flux import *
import numpy as np
import random
import re
import seaborn as sns
import matplotlib.pyplot as plt
import os
import math
from equilibrator_api import ComponentContribution, Q_, Reaction
import cvxpy as cp

## 2. Model and Data Loading
Load the C4 metabolic model and the dataset containing flux scan results.

In [2]:
c4_model = read_sbml_model("../Models/c4_model.xml")

Set parameter Username
Academic license - for non-commercial use only - expires 2027-01-20


In [3]:
df_Variable = pd.read_csv("../Datasets/Scan_Variable_Weighted.csv")

In [4]:
df_variable_structured = structure_data(df=df_Variable, model=c4_model)

In [5]:
df_variable_structured

,Vc_Vo,PPFD,V_BS_M,Maximum Nitrogen Uptake,Maximum Carbon Uptake,Carbon Uptake,Nitrogen Uptake,Light Mesophyll,Light Bundle Sheath,PC_M,...,Nitrate_reductase_M,Nitrate_reductase_B,Nitrite_reductase_M,Nitrite_reductase_B,Glutamine_syn_c_M,Glutamine_syn_m_M,Glutamine_syn_p_M,Glutamine_syn_c_B,Glutamine_syn_m_B,Glutamine_syn_p_B
0,50.0,1000,0.1,1000.0,20.0,20.0,3.7,233.363636,90.909091,1.0,...,3.727273,0.000000,3.727273,0.000000,0.0,0.000000,4.272727,0.0,0.0,0.000000
1,55.0,1000,0.1,1000.0,20.0,20.0,3.7,386.909091,90.909091,1.0,...,3.727273,0.000000,3.727273,0.000000,0.0,0.000000,10.727273,0.0,0.0,0.263636
2,60.0,1000,0.1,1000.0,20.0,20.0,3.7,419.181818,55.754545,1.0,...,3.727273,0.000000,3.727273,0.000000,0.0,0.090909,11.545455,0.0,0.0,0.281818
3,65.0,1000,0.1,1000.0,20.0,20.0,3.7,396.727273,30.990909,1.0,...,3.727273,0.000000,3.727273,0.000000,0.0,0.090909,10.272727,0.0,0.0,0.136364
4,70.0,1000,0.1,1000.0,20.0,20.0,3.7,373.090909,16.772727,1.0,...,3.727273,0.000000,3.727273,0.000000,0.0,0.090909,8.909091,0.0,0.0,0.045455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,80.0,1000,2.0,1000.0,20.0,20.0,3.7,231.500000,76.133333,1.0,...,0.000000,3.733333,2.200000,1.533333,0.0,0.000000,3.833333,0.0,0.0,1.600000
216,85.0,1000,2.0,1000.0,20.0,20.0,3.7,238.533333,57.600000,1.0,...,0.000000,3.733333,3.700000,0.000000,0.0,0.000000,3.633333,0.0,0.0,1.466667
217,90.0,1000,2.0,1000.0,20.0,20.0,3.7,224.766667,56.066667,1.0,...,1.933333,1.733333,3.400000,0.266667,0.0,0.000000,3.366667,0.0,0.0,1.200000
218,95.0,1000,2.0,1000.0,20.0,20.0,3.7,231.466667,35.400000,1.0,...,3.400000,0.266667,3.400000,0.266667,0.0,0.000000,3.466667,0.0,0.0,0.533333


## 3. Flux Normalization
Normalize flux values based on the volume ratio ($V_{BS}/V_{M}$) encoded in the column headers. This accounts for the anatomical differences between Bundle Sheath (BS) and Mesophyll (M) cells.

In [6]:
df = df_Variable

reaction_col = df.columns[0]
reactions = df[reaction_col].astype(str)

flux_cols = df.columns.tolist()[1:]

# Function to extract the volume ratio (v_bs_m) from a column name
def extract_volume_ratio(col_name):
    # Expecting something like "1500_0.1_1_1_50.0_20_1000" -> take second field after first underscore
    parts = str(col_name).split('_')
    try:
        return float(parts[1])
    except Exception:
        # fallback: regex search for a number after first underscore
        m = re.search(r'^[^_]*_(\d+(\.\d+)?)', str(col_name))
        return float(m.group(1)) if m else None

# Build a mapping of column -> volume_ratio and normalized volumes
col_info = []
for col in flux_cols:
    vr = extract_volume_ratio(col)
    if vr is None:
        vm_norm = None
        vbs_norm = None
    else:
        # Calculate normalization factors based on volume ratio
        vm_norm = 1.0 / (1.0 + vr)
        vbs_norm = vr / (1.0 + vr)
    col_info.append({
        "column": col, 
        "volume_ratio": vr, 
        "VM_Norm": vm_norm, 
        "VBS_Norm": vbs_norm
    })

col_info_df = pd.DataFrame(col_info)

# Pre-identify reaction types
is_mes = reactions.str.startswith("[M]")
is_bs = reactions.str.startswith("[B]")

# Check if reaction ends with "_tx" but is NOT a Photon_tx reaction
is_tx_non_photon = (
    reactions.str.endswith("_tx") & 
    ~reactions.isin(["[M]_Photon_tx", "[B]_Photon_tx"])
)

# Process all flux columns at once - build list of Series
adjusted_cols = []

for info in col_info:
    col = info["column"]
    vm = info["VM_Norm"]
    vbs = info["VBS_Norm"]

    # Copy original column as numeric (coerce errors to NaN)
    vals = pd.to_numeric(df[col], errors="coerce")

    # Determine if we should skip volume scaling for this reaction-column combination
    # Skip if: reaction ends with "_tx" AND is not a Photon_tx reaction
    skip_scaling = is_tx_non_photon
    
    # Apply scaling
    scaled = vals.copy()
    if vm is not None and vbs is not None:
        # Only scale reactions that should be scaled (non-tx or Photon_tx reactions)
        scale_mask_mes = is_mes & ~skip_scaling
        scale_mask_bs = is_bs & ~skip_scaling
        
        scaled.loc[scale_mask_mes] = vals.loc[scale_mask_mes] * vm
        scaled.loc[scale_mask_bs] = vals.loc[scale_mask_bs] * vbs
    
    adjusted_cols.append(scaled)

# Combine all columns at once using pd.concat (avoids fragmentation)
adjusted_df = pd.concat(
    [df[[reaction_col]]] + adjusted_cols,
    axis=1,
    keys=[reaction_col] + flux_cols
)

# Reset column names to match original structure
adjusted_df.columns = [reaction_col] + flux_cols

print("Shape:", adjusted_df.shape)
print("Number of flux columns:", len(flux_cols))

Shape: (1995, 221)
Number of flux columns: 220


## 4. Classification of Photosynthetic Modes
Filter the dataset to isolate flux solutions that do not conform to standard C3, C4, or intermediate definitions.

In [7]:
df_unidentified = df_variable_structured[ df_variable_structured["Photosynthesis"] == "Unidentified" ]
df_all_identified = df_variable_structured[ df_variable_structured["Photosynthesis"] != "Unidentified" ]

In [8]:
df_Variable.index = df_Variable["Unnamed: 0"]
df_Variable.drop(columns=["Unnamed: 0"], inplace=True)

In [9]:
def filter_unidentified_flux_solutions(df):
    """
    Filters flux solutions to only keep those classified as 'Unidentified'
    
    Parameters:
    df (pd.DataFrame): Original dataset with flux solutions as columns
    
    Returns:
    pd.DataFrame: Filtered DataFrame containing only 'Unidentified' flux solutions
    """
    
    def photosynthetic_classification_single_column(column_data, column_name):
        """
        Classify a single flux solution column based on key enzymatic activities.
        """
        # Filtering as a percentage of CO2 assimilation
        CO2_filter = float(column_name.split("_")[5]) * 0.1  # filter by percentage of CO2 assimilation
        VM = 1 / (1 + float(column_name.split("_")[1]))
        VBS = float(column_name.split("_")[1]) / (1 + float(column_name.split("_")[1]))
        
        # Apply filtering to this column (zero out negligible fluxes)
        filtered_column = column_data.copy()
        for line in filtered_column.index:
            if "[B]" in line:
                if -abs(CO2_filter) <= filtered_column.loc[line] * VBS <= CO2_filter:
                    filtered_column.loc[line] = 0
            elif "[M]" in line:
                if -abs(CO2_filter) <= filtered_column.loc[line] * VM <= CO2_filter:
                    filtered_column.loc[line] = 0
        
        # Extract fluxes for key marker reactions
        M_rubisco = filtered_column.get("[M]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
        B_rubisco = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
        B_rubisco_ex = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p_Ex", 0)
        B_pepck = filtered_column.get("[B]_PEPCARBOXYKIN_RXN_c", 0)
        B_malic_nadp = filtered_column.get("[B]_MALIC_NADP_RXN_p", 0)
        B_139_rxn = filtered_column.get("[B]_1_PERIOD_1_PERIOD_1_PERIOD_39_RXN_m", 0)
        M_pepcarbox = filtered_column.get("[M]_PEPCARBOX_RXN_c", 0)
        B_gcvmulti = filtered_column.get("[B]_GCVMULTI_RXN_m", 0)
        
        # Classification logic
        if (M_rubisco != 0 or B_rubisco_ex != 0) and B_rubisco == 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0 and B_gcvmulti == 0:
            return "C3"
        elif M_rubisco != 0 and B_rubisco != 0 and B_gcvmulti != 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0:
            return "C3_C4_Type_I"
        elif M_rubisco != 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0) and B_gcvmulti != 0:
            return "C3_C4_Type_II"
        elif M_rubisco == 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0):
            return "C4"
        else:
            return "Unidentified"
    
    # Apply classification to each column and filter
    unidentified_columns = []
    identified_columns = []
    
    for column in df.columns:
        classification = photosynthetic_classification_single_column(df[column], column)
        if classification == "Unidentified":
            unidentified_columns.append(column)
        else:
            identified_columns.append(column)
    
    # Return only the unidentified columns
    return df[unidentified_columns], df[identified_columns]

# Alternative function if you want to process columns one by one
def is_unidentified_flux_solution(column_data, column_name):
    """
    Check if a single flux solution column is classified as 'Unidentified'
    
    Parameters:
    column_data (pd.Series): Single column of flux data
    column_name (str): Name of the column
    
    Returns:
    bool: True if classified as 'Unidentified', False otherwise
    """
    
    # Filtering as a percentage of CO2 assimilation
    CO2_filter = float(column_name.split("_")[5]) * 0.1
    VM = 1 / (1 + float(column_name.split("_")[1]))
    VBS = float(column_name.split("_")[1]) / (1 + float(column_name.split("_")[1]))
    
    # Apply filtering
    filtered_column = column_data.copy()
    for line in filtered_column.index:
        if "[B]" in line:
            if -abs(CO2_filter) <= filtered_column.loc[line] * VBS <= CO2_filter:
                filtered_column.loc[line] = 0
        elif "[M]" in line:
            if -abs(CO2_filter) <= filtered_column.loc[line] * VM <= CO2_filter:
                filtered_column.loc[line] = 0
    
    # Classification logic (same as above)
    M_rubisco = filtered_column.get("[M]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
    B_rubisco = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p", 0)
    B_rubisco_ex = filtered_column.get("[B]_RIBULOSE_BISPHOSPHATE_CARBOXYLASE_RXN_p_Ex", 0)
    B_pepck = filtered_column.get("[B]_PEPCARBOXYKIN_RXN_c", 0)
    B_malic_nadp = filtered_column.get("[B]_MALIC_NADP_RXN_p", 0)
    B_139_rxn = filtered_column.get("[B]_1_PERIOD_1_PERIOD_1_PERIOD_39_RXN_m", 0)
    M_pepcarbox = filtered_column.get("[M]_PEPCARBOX_RXN_c", 0)
    B_gcvmulti = filtered_column.get("[B]_GCVMULTI_RXN_m", 0)
    
    # Check if unidentified
    if (M_rubisco != 0 or B_rubisco_ex != 0) and B_rubisco == 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0 and B_gcvmulti == 0:
        return False  # C3
    elif M_rubisco != 0 and B_rubisco != 0 and B_gcvmulti != 0 and B_pepck == 0 and B_malic_nadp == 0 and B_139_rxn == 0:
        return False  # C3_C4_Type_I
    elif M_rubisco != 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0) and B_gcvmulti != 0:
        return False  # C3_C4_Type_II
    elif M_rubisco == 0 and M_pepcarbox != 0 and B_rubisco != 0 and (B_pepck != 0 or B_malic_nadp != 0 or B_139_rxn != 0):
        return False  # C4
    else:
        return True  # Unidentified

In [10]:
filtered_df, all_identified = filter_unidentified_flux_solutions(df_Variable)

## 5. Identification of CO2 Producing Reactions
Analyze the "Unidentified" solutions to find which reactions are producing CO2 in the Bundle Sheath, potentially indicating novel decarboxylation pathways.

In [11]:
def budget_plot(solution_frame, metabolite_query, cell_type):
    """
    Identifies producers and consumers of a specific metabolite in a given cell type.
    """
    #Defining the list of metabolites to search
    budget_metabolites = []

    #Defining the compartments within the model
    compartment = ["_c", "_m", "_p", "_x", "_e", "_i", "_v", "_l", "tx", "ss", "pc", "xc", "mc", "ep", 'mi']

    #Searching for the metabolites in the model
    for met in c4_model.metabolites.query(metabolite_query):
        if met.id[:3] == cell_type:
            budget_metabolites.append(met)

    #Defining list of reactions producing and consuming the metabolite
    consumers = []
    producers = []

    #Add reactions to respective list and exclude [B]_L_GLUTAMATE_GAMMA_SEMIALDEHYDE_m transport reactions
    for met in budget_metabolites:
        for reaction in c4_model.reactions:
            if met in reaction.reactants and reaction.id[-2:] in compartment:
                consumers.append(reaction.id)
            elif met in reaction.products and reaction.id[-2:] in compartment:
                producers.append(reaction.id)
    
    #Get flux values from the simulation for metabolite consuming/producing reactions
    producers_df = solution_frame.loc[producers,:]
    consumers_df = solution_frame.loc[consumers,:]

    #Get values with negative flows: producing reactions with negative flow are consuming and vice-versa
    negative_producers = list(producers_df[producers_df["fluxes"] < 0].index)
    negative_consumers = list(consumers_df[consumers_df["fluxes"] < 0].index)


    #Add reactions to correct list
    consumers.extend(negative_producers)
    producers.extend(negative_consumers)

    #Remove reactions with negative flux from old list
    def remove_items(test_list, item):
        res = [i for i in test_list if i != item]
        return res

    for item in negative_producers:
        producers = remove_items(producers, item)

    for item in negative_consumers:
        consumers = remove_items(consumers, item)

    #Get flux values from the simulation for metabolite consuming/producing reactions (correct list)
    producers_df = solution_frame.loc[producers,:]
    #Make all values positive (disregard directionality)

    producers_df["fluxes"] = producers_df["fluxes"].abs()
    #Remove reactions with zero flux
    producers_df = producers_df[(producers_df.T != 0).all()]

    #Get flux values from the simulation for metabolite consuming/producing reactions (correct list)
    consumers_df = solution_frame.loc[consumers,:]

    #Make all values positive (disregard directionality)
    consumers_df["fluxes"]  = consumers_df["fluxes"].abs()
    #Remove reactions with zero flux3_P_SERINE_
    consumers_df = consumers_df[(consumers_df.T != 0).all()]

    producers_df["Status"] = "Producer"
    consumers_df["Status"] = "Consumer"

    frame = [producers_df, consumers_df]

    all_reactions = pd.concat(frame)

    all_reactions["label"] = all_reactions.index

    #Export full dataframe to csv
    #all_reactions.to_csv("budget_plot.csv", header=True)

    #Correct Budget stoichiometry
    atp_stoi=[]
    for i in all_reactions['label']:
        for met in budget_metabolites:
            try:
                rxn=c4_model.reactions.get_by_id(i).get_coefficient(met)
                atp_stoi.append(abs(rxn))
            except KeyError:
                continue

    new_flux = all_reactions["fluxes"] * atp_stoi

    all_reactions.insert(3,'coefficient', atp_stoi)
    all_reactions.insert(4,'new_flux', new_flux)

    return producers_df

In [12]:
# Create dictionaries to store results
budget_results = {}
all_co2_producers = set()
recurring_co2_producers = {}
universal_co2_producers = set()  # For reactions that appear in ALL flux solutions

CO2_List = ["CARBON_DIOXIDE_c", "CARBON_DIOXIDE_m", "CARBON_DIOXIDE_p"]

# Track which reactions appear in which columns
reaction_to_columns = {}

def is_transporter(reaction_id):
    """Check if a reaction is a transporter (ends with mc or pc)"""
    return reaction_id.endswith('_mc') or reaction_id.endswith('_pc')

for column_name in filtered_df.columns:
    column_co2_producers = set()
    
    for CO2_metabolite in CO2_List:
        # Convert the column to a DataFrame with 'fluxes' column
        column_data = filtered_df[column_name]
        solution_frame = pd.DataFrame({
            'fluxes': column_data
        })
        
        # Run budget_plot for this column and CO2 metabolite
        try:
            producers_df = budget_plot(
                solution_frame=solution_frame,
                metabolite_query=CO2_metabolite,
                cell_type="[B]"
            )
            
            # Get the unique reaction IDs from this analysis
            current_producers = set(producers_df.index)
            
            # FILTER OUT TRANSPORTERS (mc and pc)
            current_producers = {rxn for rxn in current_producers if not is_transporter(rxn)}
            
            # Add to the global set of all unique CO2 producers
            all_co2_producers.update(current_producers)
            
            # Add to this column's producers
            column_co2_producers.update(current_producers)
            
            # Update frequency count for recurring producers
            for reaction_id in current_producers:
                recurring_co2_producers[reaction_id] = recurring_co2_producers.get(reaction_id, 0) + 1
                
                # Track which columns each reaction appears in
                if reaction_id not in reaction_to_columns:
                    reaction_to_columns[reaction_id] = set()
                reaction_to_columns[reaction_id].add(column_name)
            
            #print(f"✓ {column_name} - {CO2_metabolite}: Found {len(current_producers)} producers (after filtering transporters)")
            
        except Exception as e:
            #print(f"✗ {column_name} - {CO2_metabolite}: {e}")
            pass
    
    # Store results for this column
    budget_results[column_name] = column_co2_producers

# Identify universal producers (appear in ALL flux solutions)
total_columns = len(filtered_df.columns)
universal_co2_producers = set()

for reaction_id, columns_set in reaction_to_columns.items():
    if len(columns_set) == total_columns:
        universal_co2_producers.add(reaction_id)

# SCENARIO 1: All unique reactions that produce CO2
print("\n" + "="*80)
print("SCENARIO 1: ALL UNIQUE CO2 PRODUCTION REACTIONS")
print("="*80)
print(f"Total unique CO2-producing reactions (excluding transporters): {len(all_co2_producers)}")

if all_co2_producers:
    print("\nUnique CO2 production reactions:")
    for i, reaction_id in enumerate(sorted(all_co2_producers), 1):
        frequency = recurring_co2_producers.get(reaction_id, 0)
        print(f"{i:3d}. {reaction_id}")

# SCENARIO 2: Recurring CO2 production reactions
print("\n" + "="*80)
print("SCENARIO 2: RECURRING CO2 PRODUCTION REACTIONS")
print("="*80)
min_occurrences = 2
recurring_reactions = {reaction: count for reaction, count in recurring_co2_producers.items() 
                      if count >= min_occurrences}
recurring_sorted = sorted(recurring_reactions.items(), key=lambda x: x[1], reverse=True)

print(f"Recurring CO2 production reactions (appear in ≥{min_occurrences} flux solutions, excluding transporters): {len(recurring_sorted)}")

# NEW: UNIVERSAL CO2 PRODUCTION REACTIONS
print("\n" + "="*80)
print("SCENARIO 3 : UNIVERSAL CO2 PRODUCTION REACTIONS")
print("="*80)
print(f"Universal CO2 producers (appear in ALL {total_columns} flux solutions, excluding transporters): {len(universal_co2_producers)}")

if universal_co2_producers:
    print("\nUniversal CO2 production reactions:")
    for i, reaction_id in enumerate(sorted(universal_co2_producers), 1):
        frequency = recurring_co2_producers.get(reaction_id, 0)
        print(f"{i:3d}. {reaction_id}")
else:
    print("No universal CO2 production reactions found (after filtering transporters).")

# Additional analysis: Show reactions that appear in almost all solutions (e.g., 80%+)
print("\n" + "="*80)
print("SCENARIO 4: NEAR-UNIVERSAL CO2 PRODUCTION REACTIONS (≥80% of solutions)")
print("="*80)

min_percentage = 0.8  # 80%
min_count = int(total_columns * min_percentage)

near_universal_reactions = []
for reaction_id, count in recurring_co2_producers.items():
    if count >= min_count:
        percentage = (count / total_columns) * 100
        near_universal_reactions.append((reaction_id, count, percentage))

# Sort by frequency
near_universal_reactions.sort(key=lambda x: x[1], reverse=True)

print(f"Near-universal CO2 producers (appear in ≥{min_percentage*100:.0f}% of solutions, excluding transporters): {len(near_universal_reactions)}")
for i, (reaction_id, count, percentage) in enumerate(near_universal_reactions, 1):
    print(f"{i:3d}. {reaction_id} ({count}/{total_columns} solutions, {percentage:.1f}%)")

# Breakdown of near-universal producers by compartment
print("\n" + "="*80)
print("NEAR-UNIVERSAL PRODUCERS BY COMPARTMENT")
print("="*80)

near_universal_by_compartment = {
    'cytosol': [],
    'mitochondria': [],
    'plastid': [],
    'other': []
}

# near_universal_reactions is a list of tuples: (reaction_id, count, percentage)
for reaction_id, count, percentage in near_universal_reactions:
    if reaction_id.endswith('_c'):
        near_universal_by_compartment['cytosol'].append((reaction_id, count, percentage))
    elif reaction_id.endswith('_m'):
        near_universal_by_compartment['mitochondria'].append((reaction_id, count, percentage))
    elif reaction_id.endswith('_p'):
        near_universal_by_compartment['plastid'].append((reaction_id, count, percentage))
    else:
        near_universal_by_compartment['other'].append((reaction_id, count, percentage))

for compartment, reactions in near_universal_by_compartment.items():
    print(f"\n{compartment.upper()}: {len(reactions)} near-universal reactions")
    for i, (reaction, count, pct) in enumerate(reactions, 1):
        print(f"  {i}. {reaction} ({count}/{total_columns}, {pct:.1f}%)")

# Save results
import json

results_to_save = {
    'all_unique_reactions': list(all_co2_producers),
    'universal_reactions': list(universal_co2_producers),
    'near_universal_reactions': near_universal_reactions,
    'recurring_reactions': recurring_co2_producers,
    'compartment_breakdown': near_universal_by_compartment,
    'total_flux_solutions': total_columns,
    'note': 'Transporters (mc and pc) have been filtered out'
}


SCENARIO 1: ALL UNIQUE CO2 PRODUCTION REACTIONS
Total unique CO2-producing reactions (excluding transporters): 21

Unique CO2 production reactions:
  1. [B]_1_PERIOD_1_PERIOD_1_PERIOD_39_RXN_m
  2. [B]_2_PERIOD_3_PERIOD_1_PERIOD_180_RXN_p
  3. [B]_ACETOLACTSYN_RXN_p
  4. [B]_ACETOOHBUTSYN_RXN_p
  5. [B]_CARBOXYCYCLOHEXADIENYL_DEHYDRATASE_RXN_p
  6. [B]_DIAMINOPIMDECARB_RXN_p
  7. [B]_GCVMULTI_RXN_m
  8. [B]_IGPSYN_RXN_p
  9. [B]_ISOCITDEH_RXN_c
 10. [B]_MALIC_NADP_RXN_p
 11. [B]_PEPCARBOXYKIN_RXN_c
 12. [B]_PYRUVDEH_RXN_m
 13. [B]_PYRUVDEH_RXN_p
 14. [B]_RXN_5682_p
 15. [B]_RXN_7800_p
 16. [B]_RXN_9648_p
 17. [B]_RXN_9650_p
 18. [B]_RXN_9651_p
 19. [B]_RXN_9652_p
 20. [B]_RXN_9653_p
 21. [B]_RXN_9654_p

SCENARIO 2: RECURRING CO2 PRODUCTION REACTIONS
Recurring CO2 production reactions (appear in ≥2 flux solutions, excluding transporters): 21

SCENARIO 3 : UNIVERSAL CO2 PRODUCTION REACTIONS
Universal CO2 producers (appear in ALL 47 flux solutions, excluding transporters): 3

Universal C

In [13]:
near_universal_reactions

[('[B]_ISOCITDEH_RXN_c', 47, 100.0),
 ('[B]_RXN_7800_p', 47, 100.0),
 ('[B]_ACETOLACTSYN_RXN_p', 47, 100.0),
 ('[B]_PYRUVDEH_RXN_m', 38, 80.85106382978722)]

In [26]:
print("\n" + "="*80)
print("Exchange reactions (≥60% of solutions)")
print("="*80)

epsilon = 0.00001 
threshold = 0.6 * len(filtered_df.columns)

active_rows_df = filtered_df[filtered_df.abs().gt(epsilon).sum(axis=1) >= threshold]

mb_active_rows = active_rows_df[active_rows_df.index.str.contains('MB]_', na=False)].copy()

mb_active_rows['Average'] = mb_active_rows.mean(axis=1)
mb_active_rows['SD'] = mb_active_rows.std(axis=1)

summary_df = mb_active_rows[['Average', 'SD']]
print(summary_df)


Exchange reactions (≥60% of solutions)
                          Average            SD
Unnamed: 0                                     
[MB]_VAL_c              -0.171855  7.888475e-02
[MB]_NITRATE_c          -3.097476  1.204704e+00
[MB]_WATER_c           -15.570408  2.228806e+00
[MB]_GLT_c              -0.498325  3.142738e-02
[MB]_MAL_c               0.559157  1.531956e+00
[MB]_L_ASPARTATE_c       0.917645  1.222533e+00
[MB]_SULFATE_c          -0.092336  6.471417e-03
[MB]_OXYGEN_MOLECULE_c  -2.536831  2.002348e+00
[MB]_LEU_c              -0.228815  2.489612e-15


## 6. Thermodynamic Analysis using Component Contribution
Use the `equilibrator_api` to calculate the standard Gibbs free energy ($\Delta G'^\circ$) and physiological Gibbs free energy ($\Delta G'$) for the identified reactions.

In [17]:
cc = ComponentContribution()

In [18]:
def parse_compounds(compound_id):
    """
    Parse reaction ID from format '[B]_REACTION_NAME_compartment' 
    to 'REACTION-NAME' format.
    
    Args:
        reaction_id (str): Reaction ID in format '[B]_NAME_compartment'
        
    Returns:
        str: Cleaned reaction ID with underscores replaced by hyphens
    """
    # Remove '[B]_' prefix
    cleaned = compound_id.replace('[B]_', '')
    
    # Remove the last underscore and everything after it (compartment suffix)
    # Find the last underscore
    last_underscore = cleaned.rfind('_')
    if last_underscore != -1:
        cleaned = cleaned[:last_underscore]
    
    # Replace remaining underscores with hyphens
    cleaned = cleaned.replace('_', '-')
    
    return cleaned

In [19]:
def set_reaction(reaction_id):
    """
    Creates an Equilibrator Reaction object from a model reaction ID.
    Maps model metabolites to MetaCyc compound IDs.
    """
    compound_ids = []
    
    for metabolite in c4_model.reactions.get_by_id(reaction_id).metabolites:
        compound_ids.append(parse_compounds(metabolite.id))
   
    compound_dict = {cid : cc.get_compound(f"metacyc.compound:{cid}") for cid in compound_ids}

    # initialize once before the loop
    compound_dict_reaction = {}
    for metabolite in c4_model.reactions.get_by_id(reaction_id).metabolites:
        key = compound_dict[parse_compounds(metabolite.id)]
        coeff = c4_model.reactions.get_by_id(reaction_id).get_coefficient(metabolite)
        compound_dict_reaction[key] = coeff  # update instead of recreating
    
    #Print reaction details with common names
    print(", ".join([f"{c.get_common_name()}: {s}" for c, s in compound_dict_reaction.items()]))
    reaction = Reaction(compound_dict_reaction)
    print("The reaction is " + ("" if reaction.is_balanced() else "not ") + "balanced")
    return reaction

In [21]:
reaction="[B]_RXN_7800_p"
print(c4_model.reactions.get_by_id(reaction).build_reaction_string())
equi_reaction = set_reaction(reaction)
dG0_prime = cc.standard_dg_prime(equi_reaction)
print(f"ΔG'° = {dG0_prime}")
dGm_prime = cc.physiological_dg_prime(equi_reaction)
print(f"ΔG'm = {dGm_prime}")

[B]_CPD_7100_p + [B]_PROTON_p --> [B]_2K_4CH3_PENTANOATE_p + [B]_CARBON_DIOXIDE_p
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
ΔG'° = (-14 +/- 4) kilojoule / mole
ΔG'm = (-31 +/- 4) kilojoule / mole


In [22]:
cc.p_h = Q_(8.0)
cc.p_mg = Q_(5.0)
cc.ionic_strength = Q_("0.2 M")
cc.temperature = Q_("298.15K")

In [23]:
reaction="[B]_RXN_7800_p"
print(c4_model.reactions.get_by_id(reaction).build_reaction_string())
equi_reaction = set_reaction(reaction)
dG0_prime = cc.standard_dg_prime(equi_reaction)
print(f"ΔG'° = {dG0_prime}")
dGm_prime = cc.physiological_dg_prime(equi_reaction)
print(f"ΔG'm = {dGm_prime}")

[B]_CPD_7100_p + [B]_PROTON_p --> [B]_2K_4CH3_PENTANOATE_p + [B]_CARBON_DIOXIDE_p
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
ΔG'° = (-11 +/- 4) kilojoule / mole
ΔG'm = (-28 +/- 4) kilojoule / mole


In [24]:
# optional: changing the aqueous environment parameters
cc.p_h = Q_(7.4)
cc.p_mg = Q_(3.0)
cc.ionic_strength = Q_("0.25M")
cc.temperature = Q_("298.15K")

In [25]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module='equilibrator_cache')

reactions_to_check = [
    '[B]_DIAMINOPIMDECARB_RXN_p', '[B]_ACETOOHBUTSYN_RXN_p', '[B]_RXN_5682_p',
    '[B]_RXN_7800_p', '[B]_ACETOLACTSYN_RXN_p', '[B]_CARBOXYCYCLOHEXADIENYL_DEHYDRATASE_RXN_p',
    '[B]_2_PERIOD_3_PERIOD_1_PERIOD_180_RXN_p', '[B]_RXN_9648_p', '[B]_RXN_9650_p',
    '[B]_RXN_9651_p', '[B]_RXN_9652_p', '[B]_RXN_9653_p', '[B]_RXN_9654_p',
    '[B]_PYRUVDEH_RXN_p', '[B]_PYRUVDEH_RXN_m', '[B]_ISOCITDEH_RXN_c'
]

# Define environments
env_map = {
    "_c": {
        'p_h': Q_(7.2),
        'p_mg': Q_(3.0),
        'ionic_strength': Q_("0.25 M"),
        'temperature': Q_("298.15 K")
    },
    "_m": {
        'p_h': Q_(7.8),
        'p_mg': Q_(2.5),
        'ionic_strength': Q_("0.3 M"),
        'temperature': Q_("298.15 K")
    },
    "_p": {
        'p_h': Q_(8.0),
        'p_mg': Q_(5.0),
        'ionic_strength': Q_("0.2 M"),
        'temperature': Q_("298.15 K")
    }
}

def apply_environment(cc, env):
    cc.p_h = env['p_h']
    cc.p_mg = env['p_mg']
    cc.ionic_strength = env['ionic_strength']
    cc.temperature = env['temperature']

results = []

for reaction_id in reactions_to_check:
    suffix = reaction_id[-2:]
    if suffix not in env_map:
        continue

    apply_environment(cc, env_map[suffix])

    try:
        equi_reaction = set_reaction(reaction_id)

        if any(c is None for c in equi_reaction.keys()):
            results.append({"Reaction": reaction_id, "dG0_prime": "Skipped", "dGm_prime": "Skipped"})
            continue

        # Calculate values
        dg0 = cc.standard_dg_prime(equi_reaction)
        dgm = cc.physiological_dg_prime(equi_reaction)

        # Check for high uncertainty (errors > 1000) 
        if dg0.error.magnitude > 1000:
            results.append({
                "Reaction": reaction_id, 
                "dG0_prime": "n/a", 
                "dGm_prime": "n/a"
            })
        else:
            results.append({
                "Reaction": reaction_id,
                "dG0_prime": f"{dg0.value.magnitude:.2f} +/- {dg0.error.magnitude:.2f}",
                "dGm_prime": f"{dgm.value.magnitude:.2f} +/- {dgm.error.magnitude:.2f}"
            })

    except Exception as e:
        results.append({"Reaction": reaction_id, "dG0_prime": "Error", "dGm_prime": str(e)})

df = pd.DataFrame(results)
print(df.to_string(index=False))

H+: -1.0, Meso-2,6-Diaminoheptanedioate: -1.0, L-Lysine: 1.0, CO2: 1.0
The reaction is balanced
H+: -1.0, Pyruvate: -1.0, 2-Oxobutanoate: -1.0, (S)-2-Aceto-2-hydroxybutanoate: 1.0, CO2: 1.0
The reaction is balanced
Nicotinamide adenine dinucleotide phosphate: -1.0, L-Arogenate: -1.0, CO2: 1.0, Nicotinamide adenine dinucleotide phosphate - reduced: 1.0, L-Tyrosine: 1.0
The reaction is balanced
H+: -1.0, 3-Carboxy-4-methyl-2-oxopentanoate: -1.0, CO2: 1.0, 4-Methyl-2-oxopentanoate: 1.0
The reaction is balanced
H+: -1.0, Pyruvate: -2.0, (S)-2-Acetolactate: 1.0, CO2: 1.0
The reaction is balanced
L-Arogenate: -1.0, H+: -1.0, L-Phenylalanine: 1.0, H2O: 1.0, CO2: 1.0
The reaction is balanced
Malonyl-[acyl-carrier protein]: -1.0, H+: -1.0, Acetyl-CoA: -1.0, Acetoacetyl-ACP: 1.0, CO2: 1.0, Coenzyme A: 1.0
The reaction is not balanced
H+: -1.0, Butyryl-[acp]: -1.0, Malonyl-CoA: -1.0, 3-Oxohexanoyl-[acyl-carrier protein]: 1.0, CO2: 1.0, Coenzyme A: 1.0
The reaction is not balanced
Malonyl-CoA: -1.